In [ ]:
# Cell 1: Clone project from GitHub
import os
import subprocess
from urllib.parse import urlparse

GITHUB_URL = "https://github.com/orbitorls/HeatShield.git"
PROJECT_DIR = "HeatShield"

# Validate URL
parsed = urlparse(GITHUB_URL)
if not parsed.netloc.endswith("github.com"):
    raise ValueError(f"Invalid GitHub URL: {GITHUB_URL}")

# Remove existing directory to avoid conflicts
if os.path.exists(PROJECT_DIR):
    print(f"Removing existing {PROJECT_DIR}...")
    !rm -rf {PROJECT_DIR}

# Clone repository
result = subprocess.run(
    ["git", "clone", GITHUB_URL, PROJECT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Git clone failed: {result.stderr}")

# Change to project directory
os.chdir(PROJECT_DIR)
print(f"Project cloned successfully to {os.getcwd()}")

In [ ]:
# Cell 2: Check GPU availability and memory
import subprocess

HAS_GPU = False
GPU_NAME = "CPU"
GPU_MEMORY = "N/A"

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        lines = result.stdout.strip().split("\n")
        for line in lines:
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 3:
                GPU_NAME = parts[0]
                total_mem = parts[1]
                used_mem = parts[2]
                HAS_GPU = True
                print(f"GPU Found: {GPU_NAME}")
                print(f"  Memory: {used_mem} / {total_mem}")
                break
    else:
        print("nvidia-smi returned error, will use CPU")
except FileNotFoundError:
    print("nvidia-smi not found, will use CPU")
except subprocess.TimeoutExpired:
    print("nvidia-smi timed out, will use CPU")
except Exception as e:
    print(f"GPU check failed: {e}, will use CPU")

if not HAS_GPU:
    print("Fallback to CPU training (slower)")
else:
    print(f"Using GPU: {GPU_NAME}")

In [ ]:
# Cell 3: Install dependencies
import os
import subprocess

# Check if requirements.txt exists
req_file = "requirements.txt"
if not os.path.exists(req_file):
    raise FileNotFoundError(
        f"{req_file} not found. Ensure you are in the project directory. "
        f"Current directory: {os.getcwd()}"
    )

# Install requirements (stream output so user sees progress)
print("Installing requirements...")
result = subprocess.run(
    ["pip", "install", "-r", req_file],
    text=True
)
if result.returncode != 0:
    print(f"Warning: pip install failed partially")
else:
    print("requirements.txt installed")

# Install additional ML packages
extra_packages = ["lightgbm", "xgboost", "optuna"]
print(f"Installing extra packages: {extra_packages}...")
result = subprocess.run(
    ["pip", "install"] + extra_packages,
    text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Failed to install extra packages")

print("All dependencies installed successfully")

## Cell 4a (OPTIONAL): Create zip on Drive for fastest download

**Run this cell ONCE to create data.zip on your Drive** (takes 5-10 min, but saves 10-30 min per future run)

```python
# Cell 4a: Create zip on Drive for fastest download (run ONCE)
import os
import time

# Mount Google Drive if not already mounted
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Check if data folder exists
DATA_SOURCE = "/content/drive/MyDrive/data"
if not os.path.exists(DATA_SOURCE):
    raise FileNotFoundError(f"Data folder not found: {DATA_SOURCE}")

# Create zip (run this ONCE on your local machine or in Colab)
ZIP_DEST = "/content/drive/MyDrive/data.zip"
print(f"Creating zip at {ZIP_DEST}...")
print("This takes 5-10 min but will save 10-30 min per future run")

t0 = time.time()
!zip -rq {ZIP_DEST} {DATA_SOURCE}
elapsed = time.time() - t0

# Check zip size
zip_size = os.path.getsize(ZIP_DEST) / (1024**3)
print(f"Zip created in {elapsed:.1f}s")
print(f"Zip size: {zip_size:.2f} GB")
print("Now Cell 4 will use FASTEST PATH (zip → extract)")
```

## Cell 4b (ALTERNATIVE): Ingest NASA POWER data directly (no auth required)

**Use this if you don't have data on Drive** (5-15 min, no auth, but lower accuracy than ERA5)

```python
# Cell 4b: Ingest NASA POWER data directly (no auth, 5-15 min)
import os
import time
from datetime import date, timedelta

# Configuration
START_DATE = "2023-01-01"  # Adjust as needed
END_DATE = (date.today() - timedelta(days=1)).isoformat()

print("=" * 60)
print("ALTERNATIVE: Ingest NASA POWER data (no auth required)")
print("=" * 60)
print(f"Date range: {START_DATE} → {END_DATE}")
print("This takes 5-15 min and requires no API keys")
print("Note: NASA POWER has lower accuracy than ERA5 but is sufficient for testing")

t0 = time.time()
# Run NASA POWER ingest (no auth, parallel across stations)
result = subprocess.run(
    ["python", "scripts/ingest_all.py", 
     "--sources", "nasa_power",
     "--start", START_DATE,
     "--end", END_DATE],
    capture_output=False,  # Show progress
    text=True
)
elapsed = time.time() - t0

if result.returncode != 0:
    raise RuntimeError(f"NASA POWER ingest failed: {result.stderr}")

print(f"NASA POWER ingest completed in {elapsed:.1f}s")

# Verify data
data_size = sum(os.path.getsize(os.path.join(dirpath, filename))
                for dirpath, dirnames, filenames in os.walk("data")
                for filename in filenames)
print(f"Total data size: {data_size / (1024**3):.2f} GB")
```

In [ ]:
# Cell 4: Download data from Google Drive (FAST: zip download → extract)
import os
import shutil
import subprocess
import time

# Mount Google Drive
print("Mounting Google Drive...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted")
except Exception as e:
    raise RuntimeError(
        f"Failed to mount Google Drive: {e}\n"
        "Ensure you are running in Google Colab environment."
    )

# Configuration
DATA_DEST = "./data"
ZIP_SOURCE = "/content/drive/MyDrive/data.zip"  # Zip file on Drive
DATA_SOURCE = "/content/drive/MyDrive/data"  # Fallback: folder on Drive

# Method 1: Download and extract zip (FASTEST)
if os.path.exists(ZIP_SOURCE):
    print("=" * 60)
    print("FASTEST PATH: Download zip from Drive → extract (5-10 min)")
    print("=" * 60)
    
    # Remove existing data directory
    if os.path.exists(DATA_DEST):
        print(f"Removing existing {DATA_DEST}...")
        shutil.rmtree(DATA_DEST)
    
    # Install gdown for fast download from Drive
    print("Installing gdown for Drive download...")
    subprocess.run(["pip", "install", "gdown"], check=True, capture_output=True)
    
    # Copy zip from Drive to local (faster than direct download from Drive)
    print(f"Copying zip from Drive...")
    t0 = time.time()
    subprocess.run(["cp", ZIP_SOURCE, "/tmp/data.zip"], check=True)
    copy_time = time.time() - t0
    print(f"Zip copied in {copy_time:.1f}s")
    
    # Extract zip
    print(f"Extracting zip file...")
    t0 = time.time()
    os.makedirs(DATA_DEST, exist_ok=True)
    subprocess.run(["unzip", "-q", "/tmp/data.zip", "-d", DATA_DEST], check=True)
    extract_time = time.time() - t0
    print(f"Zip extracted in {extract_time:.1f}s")
    print(f"Total time: {copy_time + extract_time:.1f}s")
    
# Method 2: Copy tar file (FAST)
elif os.path.exists("/content/drive/MyDrive/data.tar.gz"):
    print("=" * 60)
    print("FAST PATH: Copy tar from Drive → extract (7-13 min)")
    print("=" * 60)
    
    TAR_SOURCE = "/content/drive/MyDrive/data.tar.gz"
    
    # Remove existing data directory
    if os.path.exists(DATA_DEST):
        print(f"Removing existing {DATA_DEST}...")
        shutil.rmtree(DATA_DEST)
    
    # Copy tar file
    print(f"Copying tar file from Drive...")
    t0 = time.time()
    subprocess.run(["cp", TAR_SOURCE, "/tmp/data.tar.gz"], check=True)
    copy_time = time.time() - t0
    print(f"Tar copied in {copy_time:.1f}s")
    
    # Extract tar
    print(f"Extracting tar file...")
    t0 = time.time()
    os.makedirs(DATA_DEST, exist_ok=True)
    subprocess.run(["tar", "-xzf", "/tmp/data.tar.gz", "-C", DATA_DEST], check=True)
    extract_time = time.time() - t0
    print(f"Tar extracted in {extract_time:.1f}s")
    print(f"Total time: {copy_time + extract_time:.1f}s")
    
# Method 3: Copy folder (SLOW - fallback)
else:
    print("=" * 60)
    print("SLOW PATH: Copy data folder from Drive (15-40 min)")
    print("Tip: Create zip or tar on Drive for faster copy")
    print("=" * 60)
    
    # Validate source data exists
    if not os.path.exists(DATA_SOURCE):
        raise FileNotFoundError(
            f"Data folder not found at: {DATA_SOURCE}\n"
            "Please upload data.zip or data/ folder to Google Drive MyDrive."
        )
    
    # Check data contents
    expected_stations = 5  # Project has 5 Thai TMD stations
    station_folders = [d for d in os.listdir(DATA_SOURCE)
                       if os.path.isdir(os.path.join(DATA_SOURCE, d)) and "station_id=" in d]
    print(f"Found {len(station_folders)} station folders")
    if len(station_folders) < expected_stations:
        print(f"Warning: Expected {expected_stations} stations, found {len(station_folders)}")
    
    # Copy data to local (SLOW: FUSE overhead + thousands of files)
    if os.path.exists(DATA_DEST):
        print(f"Removing existing {DATA_DEST}...")
        shutil.rmtree(DATA_DEST)
    
    print("Copying data from Drive...")
    t0 = time.time()
    try:
        shutil.copytree(DATA_SOURCE, DATA_DEST)
        elapsed = time.time() - t0
        print(f"Data copied in {elapsed:.1f}s")
    except Exception as e:
        raise RuntimeError(f"Failed to copy data: {e}")

# Verify data integrity
data_size = sum(os.path.getsize(os.path.join(dirpath, filename))
                for dirpath, dirnames, filenames in os.walk(DATA_DEST)
                for filename in filenames)
print(f"Total data size: {data_size / (1024**3):.2f} GB")

In [ ]:
# Cell 5: Verify setup and project integrity
import os

REQUIRED_DIRS = ["scripts", "data", "app", "logs"]
REQUIRED_SCRIPTS = [
    "scripts/auto_tune_models.py",
    "scripts/train_forecast.py",
    "scripts/check_model_status.py"
]

print("=" * 60)
print("Verifying project setup...")
print("=" * 60)

# Check directories
missing_dirs = []
for d in REQUIRED_DIRS:
    if os.path.exists(d):
        print(f"OK {d}/")
    else:
        missing_dirs.append(d)
        print(f"MISSING {d}/ NOT FOUND")

if missing_dirs:
    raise FileNotFoundError(
        f"Missing required directories: {missing_dirs}\n"
        "Ensure the repository cloned correctly."
    )

# Check critical scripts
missing_scripts = []
for script in REQUIRED_SCRIPTS:
    if os.path.exists(script):
        print(f"OK {script}")
    else:
        missing_scripts.append(script)
        print(f"MISSING {script} NOT FOUND")

if missing_scripts:
    raise FileNotFoundError(f"Missing required scripts: {missing_scripts}")

# Check data contents
data_dirs = [d for d in os.listdir("data") if os.path.isdir(f"data/{d}")]
print(f"\nData contents: {len(data_dirs)} items")
if data_dirs:
    print(f"  Sample: {data_dirs[:3]}{'...' if len(data_dirs) > 3 else ''}")

# Check app structure
MODELS_DIR = "app/models/forecast_v3"
if os.path.exists(MODELS_DIR):
    print(f"OK Model output directory exists ({MODELS_DIR})")
else:
    os.makedirs(MODELS_DIR, exist_ok=True)
    print(f"OK Created model output directory ({MODELS_DIR})")

# Ensure logs directory exists
os.makedirs("logs", exist_ok=True)

print("\n" + "=" * 60)
print("Project setup verified")
print("=" * 60)

In [ ]:
# Cell 6: Test GPU training with LightGBM
import lightgbm as lgb
import numpy as np
import time

# Check if GPU flag exists from Cell 2
try:
    HAS_GPU
except NameError:
    HAS_GPU = False
    print("GPU flag not set, will attempt GPU test anyway")

# Generate test data
np.random.seed(42)
X = np.random.rand(2000, 30).astype("float32")
y = np.random.rand(2000).astype("float32")
train_data = lgb.Dataset(X, label=y)

# Test GPU first if available
gpu_works = False
if HAS_GPU:
    print("Testing GPU training...")
    gpu_params = {
        "device": "gpu",
        "objective": "regression",
        "metric": "rmse",
        "verbose": -1,
        "num_leaves": 31
    }
    try:
        start = time.time()
        model = lgb.train(gpu_params, train_data, num_boost_round=50)
        elapsed = time.time() - start
        print(f"GPU training works! (50 rounds in {elapsed:.2f}s)")
        gpu_works = True
        DEVICE = "gpu"
    except Exception as e:
        print(f"GPU training failed: {e}")
        print("  Falling back to CPU...")

# Test CPU if GPU not available or failed
if not gpu_works:
    print("Testing CPU training...")
    cpu_params = {
        "objective": "regression",
        "metric": "rmse",
        "verbose": -1,
        "num_leaves": 31
    }
    try:
        start = time.time()
        model = lgb.train(cpu_params, train_data, num_boost_round=50)
        elapsed = time.time() - start
        print(f"CPU training works (50 rounds in {elapsed:.2f}s)")
        DEVICE = "cpu"
    except Exception as e:
        raise RuntimeError(f"Both GPU and CPU training failed: {e}")

# Store device for later cells
print(f"\nTraining device set to: {DEVICE}")
print("This will be used for model training")

In [ ]:
# Cell 7: Train all models with auto-tuning, checkpointing, and monitoring
import os
import json
import time
import subprocess

# Configuration
MAX_ROUNDS = 5
CHECKPOINT_FILE = "logs/colab_checkpoint.json"
AUTO_TUNE_STATE = ".auto_tune_state.json"

# Get device from Cell 6
try:
    DEVICE
except NameError:
    DEVICE = "gpu"
    print("DEVICE not set, defaulting to gpu")

# Validate training script exists
TRAIN_SCRIPT = "scripts/auto_tune_models.py"
if not os.path.exists(TRAIN_SCRIPT):
    raise FileNotFoundError(f"Training script not found: {TRAIN_SCRIPT}")

# Load checkpoint if exists
completed_rounds = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        checkpoint = json.load(f)
    completed_rounds = checkpoint.get("completed_rounds", 0)
    print(f"Notebook checkpoint: {completed_rounds}/{MAX_ROUNDS} rounds completed")
    print(f"   Last update: {checkpoint.get('timestamp', 'unknown')}")

# Also check auto-tune's own state file for resume info
if os.path.exists(AUTO_TUNE_STATE):
    with open(AUTO_TUNE_STATE, "r") as f:
        at_state = json.load(f)
    print(f"Auto-tune state: {len(at_state)} models have been retried")

# Monitor GPU during training
def check_gpu_memory():
    """Check GPU memory usage."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            parts = result.stdout.strip().split(",")
            if len(parts) >= 3:
                used_mb = float(parts[0])
                total_mb = float(parts[1])
                util_pct = float(parts[2])
                return f"GPU Memory: {used_mb:.0f}/{total_mb:.0f} MB ({util_pct:.0f}% util)"
    except Exception:
        pass
    return "GPU status unavailable"

# Run training with progress monitoring
print("=" * 60)
print(f"Starting training: {MAX_ROUNDS} rounds")
print(f"GPU status: {check_gpu_memory()}")
print("=" * 60)

start_time = time.time()

try:
    # Enable quick mode for faster Colab training
    os.environ["HEATSHIELD_QUICK_MODE"] = "1"

    # Run auto-tuning (auto_tune_models.py handles device auto-detection internally)
    cmd = [
        "python", TRAIN_SCRIPT,
        "--max-rounds", str(MAX_ROUNDS),
    ]

    result = subprocess.run(
        cmd,
        capture_output=False,
        text=True
    )

    # Save checkpoint on success
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({
            "completed_rounds": MAX_ROUNDS,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "device": DEVICE,
            "status": "completed"
        }, f, indent=2)

    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed/3600:.2f} hours")
    print(f"  Final GPU status: {check_gpu_memory()}")

except KeyboardInterrupt:
    elapsed = time.time() - start_time
    print(f"\nTraining interrupted after {elapsed/3600:.2f} hours")
    # Save partial checkpoint
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({
            "completed_rounds": completed_rounds,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "device": DEVICE,
            "status": "interrupted",
            "elapsed_hours": elapsed / 3600
        }, f, indent=2)
    print("  Checkpoint saved. You can resume later.")
    raise

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\nTraining failed after {elapsed/3600:.2f} hours: {e}")
    raise

In [ ]:
# Cell 8: Check training results and model status
import os
import json
import subprocess

STATUS_SCRIPT = "scripts/check_model_status.py"
STATUS_JSON = "logs/model_status.json"

# Run status check
print("Checking model status...")
if not os.path.exists(STATUS_SCRIPT):
    raise FileNotFoundError(f"Status script not found: {STATUS_SCRIPT}")

result = subprocess.run(
    ["python", STATUS_SCRIPT],
    capture_output=False,
    text=True
)

# Parse results if JSON exists
if os.path.exists(STATUS_JSON):
    with open(STATUS_JSON, "r") as f:
        status = json.load(f)
    
    total = len(status)
    ready = sum(1 for m in status.values() if m.get("registry") == "ready")
    failed = total - ready
    
    print("\n" + "=" * 60)
    print("MODEL STATUS SUMMARY")
    print("=" * 60)
    print(f"Total models:   {total}")
    print(f"Ready:          {ready} ({ready/total*100:.1f}%)")
    print(f"Failed/Need work: {failed}")
    
    if failed > 0:
        print(f"\n{failed} models need retraining")
        print("  Run Cell 7 again to continue training")
    else:
        print("\nAll models ready!")
    print("=" * 60)
else:
    print("Model status JSON not found - training may not have completed")

In [ ]:
# Cell 9: Download trained models
import os
import shutil

MODELS_DIR = "app/models/forecast_v3"
OUTPUT_ZIP = "trained_models.zip"

# Validate models exist
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(
        f"Models directory not found: {MODELS_DIR}\n"
        "Ensure training completed successfully before downloading."
    )

# Check models directory size
total_size = sum(
    os.path.getsize(os.path.join(dirpath, filename))
    for dirpath, dirnames, filenames in os.walk(MODELS_DIR)
    for filename in filenames
)
print(f"Models directory size: {total_size / (1024**2):.1f} MB")

# Count model files
model_files = []
for dirpath, dirnames, filenames in os.walk(MODELS_DIR):
    for f in filenames:
        if f.endswith(".txt") or f.endswith(".json"):
            model_files.append(os.path.join(dirpath, f))
print(f"Model files found: {len(model_files)}")

if len(model_files) == 0:
    raise ValueError("No model files found. Training may have failed.")

# Remove existing zip if present
if os.path.exists(OUTPUT_ZIP):
    os.remove(OUTPUT_ZIP)
    print(f"Removed existing {OUTPUT_ZIP}")

# Create zip archive
print(f"Creating {OUTPUT_ZIP}...")
try:
    shutil.make_archive("trained_models", "zip", MODELS_DIR)
    zip_size = os.path.getsize(OUTPUT_ZIP)
    print(f"Archive created: {zip_size / (1024**2):.1f} MB")
except Exception as e:
    raise RuntimeError(f"Failed to create zip archive: {e}")

# Download
print("Starting download...")
try:
    from google.colab import files
    files.download(OUTPUT_ZIP)
    print("Download initiated. Check your browser's Downloads folder.")
except Exception as e:
    print(f"Download failed: {e}")
    print("  Alternative: Copy zip to Google Drive (see Cell 10)")

In [ ]:
# Cell 10: Save models to Google Drive (backup)
import os
import json
import shutil
import time

# Mount Drive if not already mounted
DRIVE_MOUNT = "/content/drive"
if not os.path.exists(DRIVE_MOUNT + "/MyDrive"):
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
else:
    print("Google Drive already mounted")

# Configuration
DRIVE_DEST = "/content/drive/MyDrive/heatshield_models"
MODELS_DIR = "app/models/forecast_v3"
STATUS_JSON = "logs/model_status.json"
CHECKPOINT_JSON = "logs/colab_checkpoint.json"

# Validate source files exist
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models not found at {MODELS_DIR}")

# Create destination directory with timestamp
timestamp = time.strftime("%Y%m%d_%H%M%S")
backup_dir = f"{DRIVE_DEST}_{timestamp}"
os.makedirs(backup_dir, exist_ok=True)
print(f"Backup directory: {backup_dir}")

# Copy models
try:
    print("Copying models...")
    shutil.copytree(MODELS_DIR, os.path.join(backup_dir, "models"))
    print("Models copied")
except Exception as e:
    raise RuntimeError(f"Failed to copy models: {e}")

# Copy status files
for src_file in [STATUS_JSON, CHECKPOINT_JSON]:
    if os.path.exists(src_file):
        dst = os.path.join(backup_dir, os.path.basename(src_file))
        shutil.copy2(src_file, dst)
        print(f"Copied {src_file}")

# Create summary
summary = {
    "backup_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": globals().get("DEVICE", "unknown"),
    "gpu": globals().get("GPU_NAME", "unknown"),
    "checkpoint": globals().get("CHECKPOINT_FILE", "none")
}
summary_path = os.path.join(backup_dir, "backup_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'=' * 60}")
print(f"Backup saved to Google Drive:")
print(f"  {backup_dir}")
print(f"{'=' * 60}")
print("\nYou can access your models anytime from Google Drive")